# SHAP Explainable Contributions in IsoTree

The [IsoTree](https://github.com/david-cortes/isotree) library does not offer any methods for feature importance or explainability of predictions, but when using parameter `ndim=1`, the library generates standard decision trees which can be exported to other formats that other libraries can consume to provide additional functionalities.

In order to determine what features drive the predicted outlierness score in isolation forests, the [SHAP](https://github.com/shap/shap) method typically used for gradient-boosted trees can be used to get an idea of how much each feature in a data point contributes towards the final score.

SHAP contributions can be calculated for TreeLite-compatible models using [daal4py](https://uxlfoundation.github.io/scikit-learn-intelex/latest/model_builders.html), which is a format supported by IsoTree, but note that it comes with several limitations as not every model type can be exported to TreeLite, and not every TreeLite model can be consumed by daal4py.

This example nevertheless ilustrates how to obtain SHAP explanations / feature contributions using this worflow.

## Building an Isolation Forest model

In [ ]:
# Data is a simple bivariate normal distribution, with no correlations
import numpy as np
n = 1000
m = 2
X = np.random.normal(size = (n, m))

### Fit a small isolation forest model
from isotree import IsolationForest
iso = IsolationForest(
    ntrees=500,
    ndim=1,
    nthreads=1,
    missing_action="impute",
    random_state=123,
).fit(X)

## Converting to daal4py through TreeLite

In [2]:
import daal4py

treelite_model = iso.to_treelite()
d4p_model = daal4py.mb.convert_model(treelite_model)

## Calculating SHAP contributions

In [3]:
d4p_model.predict(X[:5], pred_contribs=True)

array([[ 0.89931591,  0.5477458 , 15.27094513],
       [-0.29659829, -0.23537923, 15.2709453 ],
       [ 1.17124901,  0.81924997, 15.2709451 ],
       [ 0.63081201,  1.00269163, 15.27094514],
       [ 0.06501208,  1.11499939, 15.27094518]])

## Understanding the contribution predictions

### Structure of the SHAP outputs

SHAP outputs will contain feature-by-feature additive contributions, matching with the order of the features in the data and adding an intercept at the end as the last column.

The outputs from SHAP are additive and their sum should match with the raw prediction from a model, which in the case of isolation forests, will correspond to the **average depth** that is predicted, not with the standardized score produced by default by `.predict()`.

Recall again that the standardized score is calculated from the average depths as follows:

$$
\text{score} = 2 ^ { \frac{-\text{depth}} {c} }
$$

Where $c$ is a constant.

This means:

* A feature that contributes **more** towards the average depth makes the point **less** of an outlier / more of an inlier.
* A feature that contributes **less** towards the average depth makes the point **more** of an outlier.

This means that features producing large negative numbers in contributions are to be interpreted as making the observation more of an outlier, and features contributing large positive numbers are to be interpreted as making the observation more of an inlier.

In [4]:
# Contributions match against the predicted average depth,
# not against the scores
iso.predict(X[:5], output="avg_depth"), d4p_model.predict(X[:5], pred_contribs=True).sum(axis=1)

(array([16.71800684, 14.73896777, 17.26144408, 16.90444882, 16.4509567 ]),
 array([16.71800684, 14.73896778, 17.26144408, 16.90444878, 16.45095665]))

Examples:

In [5]:
# (0, 0) is the center of the distribution
# Both features should make the observation an inlier,
# meaning they should contribute positively towards depth
d4p_model.predict(
    np.array([[0., 0.]]),
    pred_contribs=True,
)

array([[ 1.02365359,  0.97887628, 15.27094509]])

In [6]:
# (3, 0) is far from the center.
# The first feature makes it more of an outlier, while the
# second one makes it more of an inlier, meaning the first
# feature should contribute negatively towards depth
d4p_model.predict(
    np.array([[3., 0.]]),
    pred_contribs=True,
)

array([[-8.36740872,  0.55215276, 15.27094575]])

In [7]:
# (-3, 0) is equally far from the center.
# Despite the sign change in the first feature, its effect
# is the same as the case above - i.e. makes the point more
# of an outlier, therefore contributes negatively towards depth
d4p_model.predict(
    np.array([[-3., 0.]]),
    pred_contribs=True,
)

array([[-8.20952944,  0.64433487, 15.27094568]])

In [ ]:
# (-3, 3) is an outlier in both features, so
# both contribute negatively towards depth
d4p_model.predict(
    np.array([[-3., 3.]]),
    pred_contribs=True,
)

array([[-5.12574153, -4.55420405, 15.27094564]])